# Most do środowisk ciągłych: od tablic do aproksymacji (TD → ćw. 0–2)

Ten notebook ma zbudować intuicję przejścia:
- **tablice** `V[s]`, `Q[s,a]` (dyskretne stany)  
- → **aproksymacja funkcji** `V̂(s; w)` (stany ciągłe)

Struktura:
1) **Ćwiczenie bazowe**: tablicowe TD(0) w małym środowisku dyskretnym
2) **Ćw. 0**: "brzydka" dyskretyzacja środowiska ciągłego i tablicowe Q-learning
3) **Ćw. 1**: liniowa aproksymacja wartości i semi-gradient TD(0)
4) **Ćw. 2**: stabilność: krok uczenia i skala cech

> W komórkach z `TODO (student)` uzupełnij fragmenty kodu.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def moving_average(x, w=25):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w)/w, mode="valid")

def plot_curve(y, title="", xlabel="episode", ylabel="value", ma_window=25):
    plt.figure()
    plt.plot(y, alpha=0.5, label="raw")
    if len(y) >= ma_window:
        plt.plot(np.arange(ma_window-1, len(y)), moving_average(y, ma_window), label=f"MA({ma_window})")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()


## 1) Ćwiczenie bazowe — tablicowe TD(0) w środowisku dyskretnym

Żeby nie zmieniać wszystkiego naraz, wracamy na chwilę do świata tablic.

Weźmiemy klasyczny **Random Walk** (Sutton & Barto): stany `1..5` są nieterminalne, `0` i `6` terminalne.
- Start w stanie `3`
- Ruch w lewo/prawo z prawdopodobieństwem 0.5 (brak akcji — to MRP)
- Nagroda `1` tylko gdy wejdziemy do terminalnego stanu `6`, w pozostałych przypadkach `0`

Dla `gamma=1` wartości prawdziwe to:  
$$v^*(i)=\frac{i}{6},\quad i\in\{1,2,3,4,5\}$$

**Twoje zadanie:** uzupełnij update TD(0) w tablicy `V` i sprawdź, czy RMSE maleje.

In [ ]:
class RandomWalkMRP:
    """Prosty MRP (bez akcji): losowy spacer po linii 0..6."""
    def __init__(self, n_states=7, start_state=3):
        assert n_states == 7, "W tym ćwiczeniu zakładamy 7 stanów: 0..6."
        self.n_states = n_states
        self.start_state = start_state
        self.state = start_state

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.state = self.start_state
        return self.state

    def step(self):
        """Zwraca (s_next, reward, done)."""
        if self.state in (0, self.n_states - 1):
            return self.state, 0.0, True

        move = -1 if np.random.rand() < 0.5 else 1
        s_next = self.state + move

        reward = 1.0 if s_next == (self.n_states - 1) else 0.0
        done = (s_next == 0) or (s_next == (self.n_states - 1))

        self.state = s_next
        return s_next, reward, done

TRUE_V = np.array([0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 0.0], dtype=float)

def td0_tabular_random_walk(env, episodes=200, alpha=0.1, gamma=1.0):
    """Tablicowe TD(0) do estymacji V(s) w MRP."""
    V = np.zeros(env.n_states, dtype=float)
    rmse_hist = []

    for ep in range(episodes):
        s = env.reset()

        done = False
        while not done:
            s_next, r, done = env.step()

            # TD target:
            # TODO (student): target = r + gamma * V[s_next] (albo samo r jeśli terminal)
            target = None  # <-- uzupełnij

            # TD error:
            # TODO (student): delta = target - V[s]
            delta = None  # <-- uzupełnij

            # Update:
            # TODO (student): V[s] += alpha * delta
            # V[s] += ...

            s = s_next

        rmse = np.sqrt(np.mean((V - TRUE_V) ** 2))
        rmse_hist.append(rmse)

    return V, rmse_hist


In [ ]:
env_rw = RandomWalkMRP()
V, rmse = td0_tabular_random_walk(env_rw, episodes=300, alpha=0.1, gamma=1.0)

print("V (learned):", np.round(V, 3))
print("V (true)   :", np.round(TRUE_V, 3))

plot_curve(rmse, title="RandomWalk: RMSE(V) podczas uczenia TD(0)", ylabel="RMSE")


### Pytania kontrolne (bazowe)

1) Czy `V` zbiega do wartości prawdziwych? Co się dzieje, gdy zmienisz `alpha` (np. 0.05 vs 0.3)?  
2) Dlaczego to działa w tablicach? (podpowiedź: każdy stan ma swoją komórkę)


## 2) Ćw. 0 — "Brzydka" dyskretyzacja stanu ciągłego i tablicowe Q-learning

Teraz robimy pierwszy krok w stronę środowisk ciągłych, ale **na siłę** zostajemy przy tablicach.

Zbudujemy proste środowisko o **ciągłym stanie** (wektor 4D), ale z **dyskretną akcją** `a∈{0,1}`.
Następnie zdyskretyzujemy stan do koszyków (binów) i spróbujemy uczyć tablicowe `Q[s,a]`.

**Cel ćwiczenia:** zobaczyć, że liczba stanów rośnie jak `B^d` (B = liczba binów na wymiar, d = wymiar stanu).


In [ ]:
class ToyCartPole:
    """Minimalne środowisko ciągłe (4D), bez zewnętrznych zależności typu gym.

    Stan: [x, x_dot, theta, theta_dot]
    Akcje: 0 -> u=-1, 1 -> u=+1
    Nagroda: 1 za każdy krok dopóki nie wypadniemy poza progi.
    """
    def __init__(self, max_steps=500, seed=0):
        self.max_steps = max_steps
        self.rng = np.random.default_rng(seed)
        self.t = 0
        self.state = None

        # Progi podobne do CartPole
        self.x_max = 2.4
        self.theta_max = 0.4

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.t = 0
        # start blisko zera
        self.state = self.rng.normal(loc=0.0, scale=0.05, size=(4,))
        return self.state.copy(), {}

    def step(self, action):
        assert action in (0, 1)
        u = -1.0 if action == 0 else 1.0
        x, xdot, th, thdot = self.state

        # Prosta liniowa dynamika + trochę szumu
        dt = 0.02
        x_next = x + dt * xdot
        xdot_next = 0.98 * xdot + 0.10 * u + self.rng.normal(0, 0.01)

        th_next = th + dt * thdot
        thdot_next = 0.98 * thdot + 0.10 * u + self.rng.normal(0, 0.01)

        self.state = np.array([x_next, xdot_next, th_next, thdot_next], dtype=float)
        self.t += 1

        terminated = (abs(x_next) > self.x_max) or (abs(th_next) > self.theta_max)
        truncated = (self.t >= self.max_steps)
        reward = 0.0 if terminated else 1.0  # jak w CartPole: 1 per step aż do fail

        return self.state.copy(), float(reward), bool(terminated), bool(truncated), {}

def epsilon_greedy(q_row, eps, rng):
    """q_row: wektor Q dla akcji w danym stanie."""
    if rng.random() < eps:
        return int(rng.integers(0, len(q_row)))
    return int(np.argmax(q_row))


**Opcjonalnie:** Jeśli masz zainstalowane `gymnasium`/`gym`, możesz w ćw. 0–2 użyć prawdziwego `CartPole-v1`.

Notebook domyślnie używa `ToyCartPole`, żeby nie wymagać dodatkowych paczek.

In [ ]:
# Opcjonalnie: użyj gym CartPole jeśli jest dostępne
USE_GYM_CARTPOLE = False  # <- ustaw na True jeśli chcesz

if USE_GYM_CARTPOLE:
    try:
        import gymnasium as gym
    except Exception:
        import gym  # type: ignore
    env_cont = gym.make("CartPole-v1")
    print("Używam gym: CartPole-v1")
else:
    env_cont = ToyCartPole(max_steps=500, seed=0)
    print("Używam ToyCartPole (wbudowane)")


In [ ]:
def make_bins(low, high, n_bins):
    """Zwraca krawędzie binów dla np.digitize."""
    return np.linspace(low, high, n_bins + 1)[1:-1]  # wewnętrzne krawędzie

def discretize_state(s, bins):
    """Mapuje stan ciągły s (4D) na krotkę indeksów (i0,i1,i2,i3).

    bins: lista 4 tablic krawędzi (dla np.digitize).
    """
    # TODO (student): użyj np.digitize osobno dla każdej składowej
    # Podpowiedź: np.digitize(x, bins) zwraca indeks w {0..n_bins}
    return None  # <-- uzupełnij

def q_learning_tabular_discretized(env, n_bins=8, episodes=300, alpha=0.2, gamma=0.99,
                                  eps_start=1.0, eps_end=0.05, eps_decay=0.995, seed=0):
    rng = np.random.default_rng(seed)

    # Dobieramy zakresy binów (heurystycznie)
    x_bins      = make_bins(-env.x_max, env.x_max, n_bins)
    xdot_bins   = make_bins(-3.0, 3.0, n_bins)
    th_bins     = make_bins(-env.theta_max, env.theta_max, n_bins)
    thdot_bins  = make_bins(-3.0, 3.0, n_bins)
    bins = [x_bins, xdot_bins, th_bins, thdot_bins]

    n_states = (n_bins + 1) ** 4
    print(f"n_bins={n_bins} -> liczba stanów ~ {(n_bins+1)}^4 = {n_states:,}")

    # Q-table jako słownik: klucz = (i0,i1,i2,i3), wartość = Q[a] dla a=0,1
    Q = {}

    def get_q(s_disc):
        if s_disc not in Q:
            Q[s_disc] = np.zeros(2, dtype=float)
        return Q[s_disc]

    ep_lengths = []
    eps = eps_start

    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        s_disc = discretize_state(s, bins)

        done = False
        t = 0
        while not done:
            q_s = get_q(s_disc)
            a = epsilon_greedy(q_s, eps, rng)

            sp, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            sp_disc = discretize_state(sp, bins)

            # TD target dla Q-learning:
            # TODO (student): target = r + gamma * max_a' Q(sp, a') (albo samo r gdy done)
            target = None  # <-- uzupełnij

            # TD error:
            # TODO (student): delta = target - Q(s,a)
            delta = None  # <-- uzupełnij

            # Update Q:
            # TODO (student): Q(s,a) += alpha * delta
            # q_s[a] += ...

            s_disc = sp_disc
            t += 1

        ep_lengths.append(t)
        eps = max(eps_end, eps * eps_decay)

    return ep_lengths, Q


In [ ]:
# Spróbuj kilku wartości n_bins i porównaj
for n_bins in [4, 8, 16]:
    lengths, Q = q_learning_tabular_discretized(env_cont, n_bins=n_bins, episodes=200, alpha=0.3, seed=0)
    plot_curve(lengths, title=f"Ćw.0: Q-learning na zdyskretyzowanym stanie (n_bins={n_bins})",
               ylabel="episode length", ma_window=20)


### Pytania kontrolne (Ćw. 0)

1) Jak rośnie liczba stanów, gdy zwiększasz `n_bins`?  
2) Czy większa liczba binów zawsze pomaga? Dlaczego uczenie może spowalniać?  
3) Ile **unikalnych** stanów faktycznie odwiedzasz (rozmiar słownika `Q`) vs ile teoretycznie istnieje?


## 3) Ćw. 1 — Liniowa aproksymacja wartości i semi-gradient TD(0)

Teraz robimy właściwy „most”: zamiast tablicy `V[s]` uczymy **parametry** `w` w funkcji:

$$\hat V(s;w)=w^T \phi(s)$$

Najprostszy wybór cech:
- $\phi(s) = [1, s_1, s_2, s_3, s_4]$ (bias + surowe cechy)

I dalej robimy **to samo TD(0)**, tylko update idzie w parametry:

$$w \leftarrow w + \alpha\,\delta\,\phi(s)$$

gdzie $\delta = r + \gamma\hat V(s';w) - \hat V(s;w)$.

W tym ćwiczeniu zostawiamy prostą politykę (np. losową), żeby skupić się na samym update.


In [ ]:
rng = np.random.default_rng(0)

def random_policy(obs, rng):
    """Losowa polityka: wybiera akcję 0/1."""
    return int(rng.integers(0, 2))

def phi(s):
    """Wektor cech φ(s)."""
    s = np.asarray(s, dtype=float)
    # TODO (student): zwróć np. [1, s1, s2, s3, s4]
    return None  # <-- uzupełnij

def v_hat(s, w):
    """V̂(s;w) = w^T φ(s)."""
    # TODO (student): policz iloczyn skalarny
    return None  # <-- uzupełnij

def td0_linear_value(env, episodes=400, alpha=0.05, gamma=0.99, max_steps=500, seed=0):
    w = np.zeros(5, dtype=float)
    returns = []

    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        G = 0.0

        for t in range(max_steps):
            a = random_policy(s, rng)
            sp, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            G += (gamma ** t) * float(r)

            # TD target
            # TODO (student): target = r + gamma * V̂(sp) (albo samo r gdy done)
            target = None  # <-- uzupełnij

            # TD error
            # TODO (student): delta = target - V̂(s)
            delta = None  # <-- uzupełnij

            # Semi-gradient update
            # TODO (student): w += alpha * delta * φ(s)
            # w += ...

            s = sp
            if done:
                break

        returns.append(G)

    return w, returns


In [ ]:
w, rets = td0_linear_value(env_cont, episodes=400, alpha=0.05, gamma=0.99, seed=0)
print("w:", np.round(w, 3))
plot_curve(rets, title="Ćw.1: Return per episode (random policy, linear TD(0) value)", ylabel="discounted return", ma_window=20)


### Pytania kontrolne (Ćw. 1)

1) Czy returny rosną? Jeśli nie — dlaczego to **nie jest** błąd? (podpowiedź: polityka jest losowa)  
2) Co w tym ćwiczeniu jest odpowiednikiem `V[s]` z tablic?  
3) Jak wygląda gradient $\nabla_w \hat V$ dla modelu liniowego?


## 4) Ćw. 2 — Stabilność: krok uczenia i skala cech

W tablicach skala stanu nie ma znaczenia (to tylko indeks).  
W aproksymacji **ma ogromne znaczenie**, bo gradient zależy od wartości cech.

W tym ćwiczeniu zrobisz mały „eksperyment numeryczny”:
- porównasz różne `alpha`
- porównasz uczenie na surowych cechach vs na cechach przeskalowanych

Proste skalowanie: dzielimy każdą składową stanu przez stałą tak, żeby typowe wartości były rzędu 1.


In [ ]:
def phi_scaled(s, scale):
    """φ(s) z prostym skalowaniem: [1, s/scale]."""
    s = np.asarray(s, dtype=float)
    return np.concatenate([[1.0], s / scale])

def td0_linear_value_with_phi(env, phi_fn, episodes=300, alpha=0.05, gamma=0.99, max_steps=500, seed=0):
    w = np.zeros(5, dtype=float)
    rng = np.random.default_rng(seed)
    returns = []
    w_norm = []

    def v_hat_local(s):
        return float(np.dot(w, phi_fn(s)))

    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        G = 0.0

        for t in range(max_steps):
            a = int(rng.integers(0, 2))
            sp, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            G += (gamma ** t) * float(r)
            target = float(r) if done else float(r) + gamma * v_hat_local(sp)
            delta = target - v_hat_local(s)

            # update
            w += alpha * delta * phi_fn(s)

            s = sp
            if done:
                break

        returns.append(G)
        w_norm.append(float(np.linalg.norm(w)))

    return w, returns, w_norm

# Skale (heurystycznie) — tu dobieramy podobnie jak w ćw. 0
scale = np.array([env_cont.x_max, 3.0, env_cont.theta_max, 3.0], dtype=float)

alphas = [0.005, 0.02, 0.05, 0.2]

results = []
for alpha in alphas:
    # surowe
    w_raw, rets_raw, norm_raw = td0_linear_value_with_phi(env_cont, lambda s: np.concatenate([[1.0], s]), alpha=alpha, episodes=250, seed=0)
    # skalowane
    w_s, rets_s, norm_s = td0_linear_value_with_phi(env_cont, lambda s: phi_scaled(s, scale), alpha=alpha, episodes=250, seed=0)

    results.append((alpha, rets_raw, norm_raw, rets_s, norm_s))

# Wykresy: surowe vs skalowane
for alpha, rets_raw, norm_raw, rets_s, norm_s in results:
    plot_curve(rets_raw, title=f"Ćw.2: returny (RAW φ), alpha={alpha}", ylabel="discounted return", ma_window=20)
    plot_curve(rets_s, title=f"Ćw.2: returny (SCALED φ), alpha={alpha}", ylabel="discounted return", ma_window=20)


### Pytania kontrolne (Ćw. 2)

1) Dla jakich `alpha` uczenie robi się niestabilne? Jak to widać na wykresach?  
2) Czy skalowanie cech pomaga? Dlaczego?  
3) Co byś zmienił(a), gdybyś chciał(a) uczyć się szybciej, ale stabilnie?


## Co dalej?

W tym notebooku zobaczyliśmy:
- tablicowe TD(0) w dyskretnym MRP,
- dlaczego dyskretyzacja środowiska ciągłego szybko staje się niewygodna,
- semi-gradient TD(0) z aproksymacją liniową,
- rolę skali cech i kroku uczenia.

**Następny krok w bloku 2:** kiedy akcje są ciągłe (albo gdy `argmax_a Q(s,a)` jest trudny), naturalnie przechodzimy do uczenia polityki bezpośrednio (policy gradient / actor-critic / PPO).
